# 05 — YOLO data overview

Scans `data/archive`, pairs every image with its YOLO annotation, validates the
annotation format and writes the manifest consumed by the preprocessing step.

The whole scan lives in `diagnostic_des_tumeurs.yolo.functions.inspection`; the
notebook only calls it and displays the report.

## 1. Build the validated manifest

`build_manifest` also saves three overview figures under
`data/processed/yolo_overview/`. The summary below shows the paired counts,
invalid annotations, mixed-class images and discarded files.

In [ ]:
from IPython.display import Image, display

from diagnostic_des_tumeurs.config import Paths
from diagnostic_des_tumeurs.yolo.functions.inspection import build_manifest

paths = Paths.from_root()
report = build_manifest(paths)
report.summary()

## 2. Paired images per folder and split

The archive uses `Train` and `Val` folders. `paired` counts images that have a
matching `.txt` annotation.

In [ ]:
report.paired_df.pivot(index="folder", columns="split", values="paired").reindex(
    report.paired_df["folder"].unique()
)

## 3. Annotation checks

An empty annotation file is valid (background image). Lines are checked for the
five YOLO fields, class range, normalised coordinates and boxes crossing the
image border. The tables list invalid annotations, mixed-class images and the four
files that cannot be paired (two images without annotation, two annotations
without image).

In [ ]:
print("Invalid annotation files:", len(report.invalid_df))
print("Images with another class in the annotation:", int((~report.consistency_df["consistent"]).sum()))
print("Missing labels:", len(report.discarded["missing_labels"]))
print("Missing images:", len(report.discarded["missing_images"]))
display(report.invalid_df)
display(report.consistency_df.loc[~report.consistency_df["consistent"]].head())
display(report.discarded["missing_labels"])
display(report.discarded["missing_images"])

## 4. Image checks and overview figures

All archive images are opened once to verify that they are readable and to
summarise their resolutions.

In [ ]:
print("Checked images:", len(report.checks_df))
print("Unreadable images:", len(report.unreadable_df))
print("Channel counts:", report.checks_df["channels"].value_counts(dropna=False).to_dict())
print("Most common resolutions:")
print(report.size_counts.head(10).to_string())

for name, path in report.figure_paths.items():
    print(name, "->", path)
    display(Image(filename=str(path)))

## 5. Manifest

The manifest stores every validated pair, its boxes, the class consistency and
the discarded files. Notebook 06 only reads this file.

In [ ]:
print("Manifest:", paths.yolo_manifest_path)
print("Paired Train:", report.totals["Train"])
print("Paired Val  :", report.totals["Val"])
print("Manifest size:", round(paths.yolo_manifest_path.stat().st_size / 1024, 1), "KB")